# Lab 07 Solution: Human-in-the-Loop

**Goal:** Build workflows that pause for human approval, allow state editing, and resume execution.

**What you'll learn:**
- `interrupt_before`: pause before a node runs
- `interrupt_after`: pause after a node runs
- `get_state()` to inspect the paused state
- `update_state()` to modify state before resuming
- `invoke(None, config)` to resume a paused workflow

In [ ]:
import os
from typing import TypedDict, Annotated
from operator import add
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

## Step 1: Basic interrupt_before

Scenario: Draft an email, then PAUSE for human review before sending.

In [ ]:
class EmailState(TypedDict):
    request: str
    draft: str
    sent: bool
    log: Annotated[list, add]

def draft_email(state: EmailState) -> dict:
    """LLM drafts an email based on the request."""
    prompt = (
        f"Draft a brief professional email (2-3 sentences) for this request:\n"
        f"{state['request']}\n"
        f"Include a subject line. Format as:\nSubject: ...\nBody: ..."
    )
    response = llm.invoke(prompt)
    draft = response.content.strip()
    print(f"  [draft_email] Draft created ({len(draft)} chars)")
    return {"draft": draft, "log": ["Email drafted"]}

def send_email(state: EmailState) -> dict:
    """Simulate sending the email."""
    print(f"  [send_email] SENDING: {state['draft'][:50]}...")
    return {"sent": True, "log": ["Email sent!"]}

In [ ]:
graph1 = StateGraph(EmailState)
graph1.add_node("draft", draft_email)
graph1.add_node("send", send_email)
graph1.add_edge(START, "draft")
graph1.add_edge("draft", "send")
graph1.add_edge("send", END)

# KEY: interrupt_before="send" — pause BEFORE sending!
memory1 = MemorySaver()
app1 = graph1.compile(checkpointer=memory1, interrupt_before=["send"])

print("Graph: START → draft → [PAUSE] → send → END")

In [ ]:
config1 = {"configurable": {"thread_id": "email-001"}}

# Run — this will PAUSE before the "send" node
result = app1.invoke(
    {"request": "Tell the team about Friday's team lunch at 1 PM", "log": []},
    config1
)

# Check the paused state
snapshot = app1.get_state(config1)
print(f"Workflow paused!")
print(f"Next node: {snapshot.next}")
print(f"Draft preview:\n{snapshot.values['draft']}\n")
print(f"Sent: {snapshot.values.get('sent', False)}")
print("→ The email was drafted but NOT sent yet!")

## Step 2: Resume the paused workflow

Human has reviewed the draft and approves it.

In [ ]:
# Resume by passing None — "continue from where you paused"
result = app1.invoke(None, config1)

print(f"After resume:")
print(f"Sent: {result['sent']}")
print(f"Log: {result['log']}")
print("→ The email was sent after human approval!")

## Step 3: Modify state before resuming

What if the human wants to EDIT the draft before sending?

In [ ]:
config2 = {"configurable": {"thread_id": "email-002"}}

# Run until pause
app1.invoke(
    {"request": "Notify the team about server maintenance on Saturday", "log": []},
    config2
)

# Inspect the draft
snapshot = app1.get_state(config2)
print(f"Original draft:\n{snapshot.values['draft']}\n")

# Human edits the draft
edited_draft = "Subject: Server Maintenance - Saturday 10 PM\nBody: Hi team, planned maintenance this Saturday at 10 PM IST. Expected downtime: 2 hours. Please save your work before 9:45 PM."

# Update the state with the human's edit
app1.update_state(config2, {"draft": edited_draft, "log": ["[HUMAN] Draft edited"]})

print(f"Human edited the draft!")
print(f"Updated draft:\n{edited_draft}\n")

# Now resume — the send node will use the edited draft
result = app1.invoke(None, config2)
print(f"Sent with edited draft: {result['sent']}")
print(f"Log: {result['log']}")
print("→ Human modified the draft, then the workflow resumed with the changes!")

## Step 4: interrupt_after — pause AFTER a node runs

In [ ]:
class ReviewState(TypedDict):
    request: str
    analysis: str
    decision: str
    log: Annotated[list, add]

def analyze_request(state: ReviewState) -> dict:
    """LLM analyzes the request."""
    prompt = (
        f"Analyze this support request and provide a brief assessment "
        f"(2 sentences) including urgency and recommended action:\n"
        f"{state['request']}"
    )
    response = llm.invoke(prompt)
    analysis = response.content.strip()
    print(f"  [analyze] {analysis[:60]}...")
    return {"analysis": analysis, "log": ["Request analyzed"]}

def make_decision(state: ReviewState) -> dict:
    """Make a final decision based on the analysis."""
    prompt = (
        f"Based on this analysis, what should we do? "
        f"Reply with exactly: APPROVE, ESCALATE, or REJECT.\n"
        f"Analysis: {state['analysis']}"
    )
    response = llm.invoke(prompt)
    decision = response.content.strip().upper()
    if decision not in ["APPROVE", "ESCALATE", "REJECT"]:
        decision = "ESCALATE"
    print(f"  [decision] → {decision}")
    return {"decision": decision, "log": [f"Decision: {decision}"]}

In [ ]:
graph2 = StateGraph(ReviewState)
graph2.add_node("analyze", analyze_request)
graph2.add_node("decide", make_decision)
graph2.add_edge(START, "analyze")
graph2.add_edge("analyze", "decide")
graph2.add_edge("decide", END)

# interrupt_after="analyze" — pause AFTER analysis is done
memory2 = MemorySaver()
app2 = graph2.compile(checkpointer=memory2, interrupt_after=["analyze"])

print("Graph: START → analyze → [PAUSE] → decide → END")

In [ ]:
config3 = {"configurable": {"thread_id": "review-001"}}

# Run — pauses AFTER analyze completes
app2.invoke({"request": "Employee requests transfer to Bangalore office", "log": []}, config3)

# Inspect the analysis before the decision is made
snapshot = app2.get_state(config3)
print(f"Paused after analysis!")
print(f"Analysis: {snapshot.values['analysis']}")
print(f"Next node: {snapshot.next}")
print(f"Decision: {snapshot.values.get('decision', 'Not yet made')}")

In [ ]:
# Human reviews the analysis and lets it proceed
print("[Human reviews analysis and approves...]")
result = app2.invoke(None, config3)
print(f"Final decision: {result['decision']}")
print(f"Log: {result['log']}")

## Step 5: Comparison — interrupt_before vs interrupt_after

In [ ]:
print("  interrupt_before=[\"send\"]     interrupt_after=[\"analyze\"]")
print("  ─────────────────────────     ──────────────────────────")
print("  Pauses BEFORE node runs       Pauses AFTER node completes")
print("  Use: approve actions           Use: review outputs")
print("  Example: send email,           Example: review analysis,")
print("           make payment                   check classification")
print("           delete data                    verify LLM output")

## TODO 1 Solution: Expense Approval Workflow

In [ ]:
class ExpenseState(TypedDict):
    employee: str
    description: str
    amount: int
    flagged: bool
    approved: bool
    reason: str
    log: Annotated[list, add]

def submit_expense(state: ExpenseState) -> dict:
    """Capture expense submission."""
    print(f"  [submit] {state['employee']}: Rs {state['amount']} — {state['description']}")
    return {"log": [f"Submitted: Rs {state['amount']} by {state['employee']}"]}

def validate_expense(state: ExpenseState) -> dict:
    """Check if expense needs manager review."""
    flagged = state["amount"] > 5000
    status = "FLAGGED for review" if flagged else "Auto-eligible"
    print(f"  [validate] Rs {state['amount']} → {status}")
    return {
        "flagged": flagged,
        "log": [f"Validated: {status}"],
    }

def approve_expense(state: ExpenseState) -> dict:
    """Mark as approved (runs after human review if flagged)."""
    # If not flagged, auto-approve
    if not state["flagged"]:
        print(f"  [approve] Auto-approved (Rs {state['amount']} <= 5000)")
        return {
            "approved": True,
            "reason": "Auto-approved: under Rs 5000 threshold",
            "log": ["Auto-approved"],
        }
    # If flagged, the human should have set approved via update_state
    status = "APPROVED by manager" if state.get("approved") else "REJECTED by manager"
    print(f"  [approve] {status}")
    return {
        "log": [f"Manager decision: {status}"],
    }

In [ ]:
graph_exp = StateGraph(ExpenseState)
graph_exp.add_node("submit", submit_expense)
graph_exp.add_node("validate", validate_expense)
graph_exp.add_node("approve", approve_expense)
graph_exp.add_edge(START, "submit")
graph_exp.add_edge("submit", "validate")
graph_exp.add_edge("validate", "approve")
graph_exp.add_edge("approve", END)

memory_exp = MemorySaver()
app_exp = graph_exp.compile(checkpointer=memory_exp, interrupt_before=["approve"])

print("Graph: START → submit → validate → [PAUSE] → approve → END")

In [ ]:
# Test 1: Small expense (auto-approve)
print("Test 1: Small expense")
config_a = {"configurable": {"thread_id": "expense-001"}}
app_exp.invoke({
    "employee": "Priya", "description": "Team lunch", "amount": 450,
    "flagged": False, "approved": False, "reason": "", "log": [],
}, config_a)

snap = app_exp.get_state(config_a)
print(f"  Paused. Flagged: {snap.values['flagged']}")
# Human approves (it's small, just let it through)
result = app_exp.invoke(None, config_a)
print(f"  Result: approved={result['approved']}, reason={result['reason']}")
print(f"  Log: {result['log']}")

In [ ]:
# Test 2: Large expense (needs manager review)
print("Test 2: Large expense")
config_b = {"configurable": {"thread_id": "expense-002"}}
app_exp.invoke({
    "employee": "Vikram", "description": "Client dinner at Taj", "amount": 8500,
    "flagged": False, "approved": False, "reason": "", "log": [],
}, config_b)

snap = app_exp.get_state(config_b)
print(f"  Paused. Flagged: {snap.values['flagged']}, Amount: Rs {snap.values['amount']}")

# Manager approves
print("  [Manager reviews and APPROVES]")
app_exp.update_state(config_b, {
    "approved": True,
    "reason": "Approved: valid client entertainment expense",
    "log": ["[MANAGER] Reviewed and approved"],
})
result = app_exp.invoke(None, config_b)
print(f"  Result: approved={result['approved']}, reason={result['reason']}")
print(f"  Log: {result['log']}")

In [ ]:
# Test 3: Large expense — manager REJECTS
print("Test 3: Large expense (rejected)")
config_c = {"configurable": {"thread_id": "expense-003"}}
app_exp.invoke({
    "employee": "Rahul", "description": "Personal shopping", "amount": 12000,
    "flagged": False, "approved": False, "reason": "", "log": [],
}, config_c)

snap = app_exp.get_state(config_c)
print(f"  Paused. Flagged: {snap.values['flagged']}, Amount: Rs {snap.values['amount']}")

# Manager rejects
print("  [Manager reviews and REJECTS]")
app_exp.update_state(config_c, {
    "approved": False,
    "reason": "Rejected: personal expense not covered by policy",
    "log": ["[MANAGER] Reviewed and rejected"],
})
result = app_exp.invoke(None, config_c)
print(f"  Result: approved={result['approved']}, reason={result['reason']}")
print(f"  Log: {result['log']}")

## TODO 2 Solution: Multi-gate Workflow (Two Pauses)

In [ ]:
class MultiGateState(TypedDict):
    request: str
    draft: str
    reviewed: bool
    sent: bool
    log: Annotated[list, add]

def create_draft(state: MultiGateState) -> dict:
    prompt = f"Draft a brief 2-sentence professional email for: {state['request']}"
    response = llm.invoke(prompt)
    draft = response.content.strip()
    print(f"  [draft] Created: {draft[:50]}...")
    return {"draft": draft, "log": ["Draft created"]}

def review_draft(state: MultiGateState) -> dict:
    print(f"  [review] Reviewing draft...")
    return {"reviewed": True, "log": ["Draft reviewed and approved"]}

def send_message(state: MultiGateState) -> dict:
    print(f"  [send] Sending: {state['draft'][:40]}...")
    return {"sent": True, "log": ["Message sent!"]}

In [ ]:
graph_mg = StateGraph(MultiGateState)
graph_mg.add_node("draft", create_draft)
graph_mg.add_node("review", review_draft)
graph_mg.add_node("send", send_message)
graph_mg.add_edge(START, "draft")
graph_mg.add_edge("draft", "review")
graph_mg.add_edge("review", "send")
graph_mg.add_edge("send", END)

memory_mg = MemorySaver()
# TWO interrupt points!
app_mg = graph_mg.compile(checkpointer=memory_mg, interrupt_before=["review", "send"])

print("Graph: START → draft → [PAUSE 1] → review → [PAUSE 2] → send → END")

In [ ]:
config_d = {"configurable": {"thread_id": "multi-001"}}

# Phase 1: Draft is created, pauses before review
app_mg.invoke({"request": "Announce team outing on Friday", "reviewed": False, "sent": False, "log": []}, config_d)
snap = app_mg.get_state(config_d)
print(f"PAUSE 1: Before review. Next: {snap.next}")
print(f"  Draft: {snap.values['draft'][:60]}...")

# Human approves review
print("\n  [Human approves for review]")
result = app_mg.invoke(None, config_d)

# Now paused again before send
snap = app_mg.get_state(config_d)
print(f"\nPAUSE 2: Before send. Next: {snap.next}")
print(f"  Reviewed: {snap.values['reviewed']}")

# Human approves sending
print("\n  [Human approves sending]")
result = app_mg.invoke(None, config_d)
print(f"\nFinal: sent={result['sent']}")
print(f"Log: {result['log']}")
print("→ Two human approvals were required before the message was sent!")

## Key Takeaways

- `interrupt_before`: pause BEFORE a node (approve actions)
- `interrupt_after`: pause AFTER a node (review outputs)
- `get_state()`: inspect the paused workflow state
- `update_state()`: modify state before resuming
- `invoke(None, config)`: resume from the pause point
- Requires a checkpointer (`MemorySaver`) to work
- TODO 1: Expense approval with flagging > Rs 5000
- TODO 2: Multi-gate with TWO `interrupt_before` points